In [1]:
!pip install cryptography

Defaulting to user installation because normal site-packages is not writeable


In [2]:
#imports
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from courbe_elliptique import multiplication_point_cf, addition_deux_points_cf, trouver_ordre_point,is_point_infini
from fct import encrypt_message
import os
import hashlib, json, time
from coincurve.utils import get_valid_secret
from eth_keys import keys
from ecies import encrypt, decrypt
import random

# Introduction à ECIES

ECIES (Elliptic Curve Integrated Encryption Scheme) est un protocole de chiffrement hybride qui combine la cryptographie à courbe elliptique (ECC) et le chiffrement symétrique pour offrir confidentialité et authenticité des messages. Il permet à un expéditeur de chiffrer un message à l'aide de la clé publique du destinataire, en générant une clé éphémère et un secret partagé, puis en utilisant une clé dérivée pour chiffrer le message avec un algorithme symétrique (comme AES-GCM). ECIES est largement utilisé dans les systèmes modernes pour sécuriser les échanges sur des réseaux publics, notamment dans les applications blockchain et les portefeuilles numériques.

## Fonctions encryptage et décryptage

ecies_encrypt : Cette fonction permet de chiffrer un message à l’aide de la clé publique du destinataire, en générant une clé éphémère et un secret partagé, puis en utilisant une clé dérivée pour le chiffrement symétrique (AES-GCM).

In [3]:
import secrets
def ecies_encrypt(message_clair, Q_B, G, a, p, n):
    """
    Chiffre le message pour Bob en utilisant ECIES 
    
    :param message_clair: Le message à chiffrer (str).
    :param Q_B: Clé publique de Bob (tuple (x, y)).
    :param G: Point générateur de la courbe (tuple (x, y)).
    :param a, p: Paramètres de la courbe (y^2 = x^3 + a*x + b mod p).
    :param n: Ordre du point G.
    :return: (Q_eph, iv, cipher_text, tag)
    """
    
    # -1: Création de la clé éphémère ECDH 
    # Alice choisit une clé privée éphémère aléatoire
    cle_priv = secrets.randbelow(n - 1) + 1
    # Elle calcule le point public éphémère
    cle_public_ephemere = multiplication_point_cf(cle_priv, G, a, p)
    
    # -2: Calcul du secret partagé 
    # Le secret partagé est S = cle_priv * Q_B (la clé publique de Bob)
    S = multiplication_point_cf(cle_priv, Q_B, a, p)
    
    # Le secret partagé pour HKDF est l'abscisse du point S
    # On convertit en bytes 
    length_in_bytes = (p.bit_length() + 7) // 8
    shared_secret_bytes = S[0].to_bytes(length_in_bytes, byteorder='big')
    
    # -3: Dérivation des clés (KDF) 
    # Utilisation d'HKDF pour dériver la clé de chiffrement (K_E)
    
    hkdf = HKDF(algorithm=hashes.SHA256(),
                length=32,
                salt=None,
                info=b'ecies-encryption-key')
    K_E = hkdf.derive(shared_secret_bytes)

    # -4: Chiffrement Symétrique (AES-256-GCM) 
    cle_public_ephemere_bytes = str(cle_public_ephemere).encode()
    
    # Appel à la fonction de fct.py (avec l'AAD)
    # *Note : Votre fct.py doit supporter l'AAD pour cette ligne.*
    iv, cipher_text, tag = encrypt_message(
        K_E, 
        message_clair, 
        aad=cle_public_ephemere_bytes
    )
    
    return cle_public_ephemere, iv, cipher_text, tag

ecies_decrypt : Elle permet au destinataire de déchiffrer le message reçu grâce à sa clé privée et la clé publique éphémère envoyée par l’expéditeur.

In [4]:
def ecies_decrypt(cle_public_ephemere, iv, cipher_text, tag, cle_priv_Bob, a, p):
    """
    Déchiffre le message ECIES.
    
    :param cle_public_ephemere: Clé publique éphémère d'Alice (tuple (x, y)).
    :param iv: Vecteur d'initialisation.
    :param cipher_text: Texte chiffré.
    :param tag: Tag d'authentification GCM.
    :param cle_priv_Bob: Clé privée de Bob.
    :param a, p: Paramètres de la courbe.
    :return: Le message déchiffré (str) ou None si l'authentification échoue.
    """
    
    # -1: Calcul du secret partagé
    # Bob calcule S' = cle_priv_Bob * cle_public_ephemere 
    S_prime = multiplication_point_cf(cle_priv_Bob, cle_public_ephemere, a, p)
    
    # Le secret partagé pour HKDF est l'abscisse du point S'
    length_in_bytes = (p.bit_length() + 7) // 8
    shared_secret_bytes = S_prime[0].to_bytes(length_in_bytes, byteorder='big')
    
    # --- Étape 2: Dérivation des clés (KDF) ---
    # Bob doit dériver EXACTEMENT la même clé K_E qu'Alice
    hkdf = HKDF(algorithm=hashes.SHA256(),
                length=32,
                salt=None,
                info=b'ecies-encryption-key')
    K_E_prime = hkdf.derive(shared_secret_bytes)

    # --- Étape 3: Déchiffrement Symétrique et Vérification MAC ---
    try:
        cipher = Cipher(algorithms.AES(K_E_prime), modes.GCM(iv, tag))
        decryptor = cipher.decryptor()
        
        # Bob doit aussi authentifier les données additionnelles (cle_public_ephemere)
        cle_public_ephemere_bytes = str(cle_public_ephemere).encode()
        decryptor.authenticate_additional_data(cle_public_ephemere_bytes)
        
        plaintext_bytes = decryptor.update(cipher_text) + decryptor.finalize()
        return plaintext_bytes.decode()
        
    except Exception as e:
        # La méthode finalize() lèvera une exception si le tag GCM est invalide.
        # Cela signifie que le message a été altéré.
        # print(f"Déchiffrement / Vérification MAC échoué: {e}")
        return None

### Exemple
Cette simulation illustre un cas d'usage réel : la confidentialité sur un registre public. Alice intègre un message secret (« mémo ») dans une transaction financière visible de tous.

-Alice chiffre le mémo avec la clé publique de Bob.

-Bob parvient à le lire grâce à sa clé privée.

-Charlie (l'espion) échoue à déchiffrer le message, prouvant que seules les métadonnées sont publiques, mais le contenu reste confidentiel.

In [5]:
import secrets

# Parametres de la courbe
a = -28
b = 23
p = 78791 
G = (43829, 70084)
n = 79168 

# Bob genere ses cles
bob_priv = secrets.randbelow(n - 1) + 1
bob_pub = multiplication_point_cf(bob_priv, G, a, p)
print(f"Bob: Cle Publique: {bob_pub}")

# Charlie (l'espion) genere ses cles
charlie_priv = secrets.randbelow(n - 1) + 1
charlie_pub = multiplication_point_cf(charlie_priv, G, a, p)
print(f"Charlie: Cle Publique: {charlie_pub}")

print("\nAlice envoie à Bob:")
message_alice = "c'est super important dis à personne mais j'ai un crush sur Charlie"
print(f"Message clair: {message_alice}")

# Alice chiffre en utilisant la cle publique de Bob
R_eph, iv, ct, tag = ecies_encrypt(message_alice, bob_pub, G, a, p, n)
print(f"Alice envoie le message chiffre (Cle ephemere + Texte chiffre).")

print("\nBob recoit et dechiffre:")
# Bob utilise sa cle privee pour dechiffrer
decrypted_bob = ecies_decrypt(R_eph, iv, ct, tag, bob_priv, a, p)

if decrypted_bob:
    print(f"SUCCES: Bob a lu: {decrypted_bob}")
else:
    print(f"ERREUR: Bob n'a pas pu lire le message.")

print("\nCharlie intercepte et essaie de déchiffrer:")
# Charlie intercepte le message et essaie de dechiffrer avec SA cle privee
decrypted_charlie = ecies_decrypt(R_eph, iv, ct, tag, charlie_priv, a, p)

if decrypted_charlie:
    print(f"ECHEC SECURITE: Charlie a lu le message: {decrypted_charlie}")
else:
    print(f"SECURITE CONFIRMEE: Charlie n'a pas pu dechiffrer le message.")

Bob: Cle Publique: (43833, 67226)
Charlie: Cle Publique: (71641, 12760)

Alice envoie à Bob:
Message clair: c'est super important dis à personne mais j'ai un crush sur Charlie
Alice envoie le message chiffre (Cle ephemere + Texte chiffre).

Bob recoit et dechiffre:
SUCCES: Bob a lu: c'est super important dis à personne mais j'ai un crush sur Charlie

Charlie intercepte et essaie de déchiffrer:
SECURITE CONFIRMEE: Charlie n'a pas pu dechiffrer le message.


Cette simulation confirme la sécurité totale d'ECIES. Le chiffrement garantit la confidentialité du secret d'Alice : bien que Charlie ait intercepté les données, il ne peut mathématiquement pas reconstruire le secret partagé nécessaire au déchiffrement. Son échec valide la robustesse du protocole (ECDH + AES-GCM) et assure que seul le détenteur légitime de la clé privée de Bob peut accéder au contenu.


## Utilisation des bibliothèques disponibles
### Exemple cryptage d'un fichier

Dans cet exemple, nous montrons comment utiliser le protocole ECIES (Elliptic Curve Integrated Encryption Scheme) pour protéger un fichier réel. ECIES est un schéma de chiffrement hybride basé sur les courbes elliptiques, permettant de chiffrer un contenu de manière sécurisée pour un destinataire précis. Le fichier original est chiffré avec la clé publique de Bob, générant un fichier .enc totalement illisible pour toute autre personne. Seul Bob, grâce à sa clé privée, peut déchiffrer et récupérer le fichier d’origine. Ce scénario illustre un cas d’usage réel de sécurité : le partage de documents sensibles dans un environnement non sécurisé.

In [6]:
bob_priv = keys.PrivateKey(get_valid_secret())
bob_priv_hex = bob_priv.to_hex()
bob_pub_hex = bob_priv.public_key.to_hex()


print("Bob PRIVATE key:", bob_priv_hex)
print("Bob PUBLIC  key:", bob_pub_hex)


def encrypt_file_ecies(input_path, output_path, pub_hex):
    with open(input_path, "rb") as f:
        data = f.read()

    encrypted = encrypt(pub_hex, data)

    with open(output_path, "wb") as f:
        f.write(encrypted)

    print(f"Fichier '{input_path}' chiffré → '{output_path}'")


def decrypt_file_ecies(input_path, output_path, priv_hex):
    with open(input_path, "rb") as f:
        encrypted = f.read()

    decrypted = decrypt(priv_hex, encrypted)

    with open(output_path, "wb") as f:
        f.write(decrypted)

    print(f"Fichier déchiffré → '{output_path}'")


input_file = "30-Day-Max-The-Healthcare-Guarantee.pdf"          
encrypted_file = "document.enc"       # fichier chiffré
decrypted_file = "document_decrypt.pdf"

# Alice chiffre le fichier pour Bob
encrypt_file_ecies(input_file, encrypted_file, bob_pub_hex)

# Bob déchiffre le fichier
decrypt_file_ecies(encrypted_file, decrypted_file, bob_priv_hex)


Bob PRIVATE key: 0x0f6bae45c1a465d87ae1d3dbf600caee3ebf7348333832094e6aec0f5b70184c
Bob PUBLIC  key: 0xa2a575c36878e1e15d05c19a701f81e346d6603b177b1a2cf2d316d7763a2faf678caf965bca7822ba1590a41f9a4501acbecebc112439de8c99bb790f1c36a6
Fichier '30-Day-Max-The-Healthcare-Guarantee.pdf' chiffré → 'document.enc'
Fichier déchiffré → 'document_decrypt.pdf'


Cet exemple démontre concrètement l'efficacité d'ECIES pour la protection de documents sensibles, comme ici un rapport médical. En transformant le fichier original en un document chiffré (.enc) totalement inexploitable pour un tiers, le protocole garantit une confidentialité absolue en cas de vol ou d'interception. Le succès du déchiffrement par Bob valide la fiabilité de cette approche hybride, prouvant que l'intégrité des données volumineuses est parfaitement préservée lors d'échanges sur des canaux non sécurisés.